In [20]:
%pip install requests beautifulsoup4 pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
from pathlib import Path
import os
import re
import time
import unicodedata
import requests

from bs4 import BeautifulSoup
from io import BytesIO
from PIL import Image

HTML = Path("index.html")
PASTA_FOTOS = Path("imagens") / "acervos"

PASTA_FOTOS.mkdir(parents=True, exist_ok=True)

print("HTML encontrado:", HTML.exists())
print("Caminho do HTML:", HTML.resolve())
print("Pasta das fotos:", PASTA_FOTOS.resolve())

HTML encontrado: True
Caminho do HTML: C:\Users\email\OneDrive\Área de Trabalho\Projetos\Acervos\index.html
Pasta das fotos: C:\Users\email\OneDrive\Área de Trabalho\Projetos\Acervos\imagens\acervos


In [22]:
def criar_nome_arquivo(nome):
    nome = unicodedata.normalize("NFD", nome)
    nome = "".join(
        c for c in nome
        if not unicodedata.combining(c)
    )
    nome = nome.lower()
    nome = re.sub(r"[^a-z0-9]+", "-", nome)
    return nome.strip("-")


def encontrar_usuario(card):
    links = card.find_all("a", href=True)

    # Primeiro tenta encontrar o perfil diretamente no endereço.
    for link in links:
        url = requests.compat.urlparse(link["href"])

        if url.netloc.lower() in (
            "x.com", "www.x.com",
            "twitter.com", "www.twitter.com"
        ):
            partes = url.path.strip("/").split("/")

            if partes and partes[0]:
                usuario = partes[0].lstrip("@")

                if usuario.lower() not in (
                    "home", "explore", "search", "i"
                ):
                    return usuario

    # Se o endereço não apontar diretamente para o perfil,
    # tenta extrair o usuário do texto do link.
    for link in links:
        resultado = re.fullmatch(
            r"@([A-Za-z0-9_]{1,15})",
            link.get_text(strip=True)
        )

        if resultado:
            return resultado.group(1)

    return None


if not HTML.exists():
    raise FileNotFoundError(
        f"Não encontrei {HTML.resolve()}. "
        "Verifique o caminho do index.html."
    )

soup = BeautifulSoup(
    HTML.read_text(encoding="utf-8"),
    "html.parser"
)

cards = soup.select(".lista_acervos .card")

print("Total de cartões encontrados:", len(cards))

for card in cards:
    titulo = card.find("h2")
    if titulo:
        nome = titulo.get_text(strip=True)
        print(f"{nome} -> @{encontrar_usuario(card)}"
              if encontrar_usuario(card)
              else f"{nome} -> usuário não identificado")

Total de cartões encontrados: 64
Poliwag -> @acervopoliwag
Gastly -> @Gastlyarquive
Lapras -> @acervo_lapras
Flareon -> @FlareonMain
Porygon -> @PorygonFiles
Dratini -> @acervodratini
Cyndaquil -> @acervocyndaquil
Ampharos -> @acervyampharos
Marill -> @acervomarill
Espeon -> @acervoespeonn
Umbreon -> @Acervoumbreon
Slowking -> @acervoslowking
Suicune -> @acervo_suicune
Tyranitar -> @acervotyranitar
Lugia -> @acervolugia
Torchic -> @pintodotorchic
Blaziken -> @AcervyBlaziken
Mudkip -> @AcerVoMudKip__
Revavroom e Registeel -> @Revavroom_DuBem
Groudon -> @acervogroudon
Luxio -> @acervoluxio
Luxray -> @acervoluxr4y
Shellos -> @acervoshellos
Garchomp -> @acervogarchomp
Leafeon -> @acervoleafon
Rotom Dex -> @AcervoRotomDex
Giratina e Shaymin -> @haterdetudo123
Oshawott -> @oshawottdivo276
Munna -> @munnaposting
Pidove -> @acervopidove
Krookodile -> @acervokroco
Scrafty -> @acervoscrafty
Zorua (Hisui) e Zoroark -> @zoruarchive
Chandelure -> @acervochandelur
Hydreigon -> @acervohydregon
Chespi

In [23]:
LIMITE_POR_EXECUCAO = 20

baixadas = 0
falhas = []
limite_atingido = False

for card in cards:
    titulo = card.find("h2")

    if not titulo:
        continue

    nome = titulo.get_text(strip=True)
    usuario = encontrar_usuario(card)

    if not usuario:
        falhas.append((nome, "Usuário não identificado"))
        continue

    arquivo_foto = (
        PASTA_FOTOS / f"{criar_nome_arquivo(nome)}.jpg"
    )

    # Não consulta novamente as fotos já salvas.
    if arquivo_foto.exists() and arquivo_foto.stat().st_size > 0:
        print(f"Já salva: {nome}")
        continue

    if baixadas >= LIMITE_POR_EXECUCAO:
        print("Limite desta execução atingido.")
        break

    url = f"https://unavatar.io/x/{usuario}"

    try:
        resposta = requests.get(
            url,
            timeout=30,
            headers={"User-Agent": "WikiAcervos/1.0"}
        )

        if resposta.status_code == 429:
            print("Limite da API atingido. Interrompendo.")
            limite_atingido = True
            break

        tipo = resposta.headers.get("Content-Type", "")

        if resposta.status_code != 200 or not tipo.startswith("image/"):
            falhas.append(
                (nome, f"HTTP {resposta.status_code}: {tipo}")
            )
            print(f"Falha ao obter foto: {nome}")
            continue

        # Confirma que o conteúdo é uma imagem válida
        # e converte para JPEG.
        with Image.open(BytesIO(resposta.content)) as imagem:
            imagem = imagem.convert("RGB")
            imagem.save(arquivo_foto, format="JPEG", quality=90)

        baixadas += 1
        print(f"Foto salva: {nome} (@{usuario})")

    except Exception as erro:
        falhas.append((nome, str(erro)))
        print(f"Erro em {nome}: {erro}")

    time.sleep(1)

print("\nNovas fotos baixadas:", baixadas)
print("Fotos com falha:", len(falhas))
print("Limite atingido:", limite_atingido)

Limite da API atingido. Interrompendo.

Novas fotos baixadas: 0
Fotos com falha: 0
Limite atingido: True


In [24]:
print("Fotos que ainda precisam ser verificadas:\n")

for nome, motivo in falhas:
    print(f"- {nome}: {motivo}")

print("\nArquivos de imagem salvos:", 
      len(list(PASTA_FOTOS.glob("*.jpg"))))

Fotos que ainda precisam ser verificadas:


Arquivos de imagem salvos: 0


In [5]:

import requests
from datetime import datetime, timezone

url = "https://unavatar.io/x/acervofloragato"

resposta = requests.get(url, timeout=30)

print("Status HTTP:", resposta.status_code)
print("Tipo de conteúdo:", resposta.headers.get("Content-Type"))
print("Limite total:", resposta.headers.get("x-rate-limit-limit"))
print("Consultas restantes:", resposta.headers.get("x-rate-limit-remaining"))
print("Valor bruto de reset:", repr(
    resposta.headers.get("x-rate-limit-reset")
))
print("Retry-After:", resposta.headers.get("Retry-After"))

reset = resposta.headers.get("x-rate-limit-reset")

if reset:
    try:
        timestamp = int(float(reset))

        # Evita converter valores absurdos ou inválidos.
        if 0 < timestamp < 4102444800:
            horario = datetime.fromtimestamp(
                timestamp, tz=timezone.utc
            )
            print("Renovação do limite (UTC):", horario)
        else:
            print("O valor de reset não parece ser um timestamp válido.")

    except (ValueError, OverflowError, OSError) as erro:
        print("Não foi possível interpretar o reset:", erro)
else:
    print("A API não informou o horário de renovação.")

if resposta.status_code == 429:
    print("\nA API ainda está limitando as consultas. Não tente baixar várias fotos.")
elif resposta.status_code == 200 and resposta.headers.get(
    "Content-Type", ""
).startswith("image/"):
    print("\nA API retornou uma imagem.")
else:
    print("\nA API não retornou uma imagem válida.")


Status HTTP: 200
Tipo de conteúdo: image/jpeg
Limite total: 25
Consultas restantes: 1
Valor bruto de reset: '1793334792182'
Retry-After: None
O valor de reset não parece ser um timestamp válido.

A API retornou uma imagem.


In [4]:
from datetime import datetime, timezone

reset = resposta.headers.get("x-rate-limit-reset")

if reset:
    try:
        # O cabeçalho retornou milissegundos.
        timestamp_segundos = int(reset) / 1000

        horario = datetime.fromtimestamp(
            timestamp_segundos,
            tz=timezone.utc
        )

        print("Renovação do limite (UTC):", horario)

    except (ValueError, OverflowError, OSError) as erro:
        print("Não foi possível interpretar o horário:", erro)

Renovação do limite (UTC): 2026-10-30 04:33:12.182000+00:00


In [3]:

from pathlib import Path
from io import BytesIO
from PIL import Image

pasta = Path("imagens/acervos")
pasta.mkdir(parents=True, exist_ok=True)

destino = pasta / "floragato.jpg"

if resposta.status_code == 200 and resposta.headers.get(
    "Content-Type", ""
).startswith("image/"):

    with Image.open(BytesIO(resposta.content)) as imagem:
        imagem.convert("RGB").save(
            destino,
            format="JPEG",
            quality=90
        )

    print("Foto salva com sucesso!")
    print("Local:", destino.resolve())

else:
    print("A resposta atual não contém uma imagem válida.")


Foto salva com sucesso!
Local: C:\Users\email\OneDrive\Área de Trabalho\Projetos\Acervos\imagens\acervos\floragato.jpg
